In [4]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

# Load the dataset (replace with your actual file path)
df = pd.read_csv("census-income.csv")

# For illustration, let's assume df is already loaded
# Example of handling categorical features (assuming your columns like 'education', 'workclass' are categorical)
categorical_columns = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']

# Handle missing values using SimpleImputer for both categorical and numerical columns
imputer_num = SimpleImputer(strategy='mean')
imputer_cat = SimpleImputer(strategy='most_frequent')

# Apply imputer to numerical and categorical columns
df[categorical_columns] = imputer_cat.fit_transform(df[categorical_columns])
df['fnlwgt'] = imputer_num.fit_transform(df[['fnlwgt']])  # Example numerical column

# Encode categorical columns
le = LabelEncoder()
for col in categorical_columns:
    df[col] = le.fit_transform(df[col])

# Assume 'Annual-Income' is the target variable for classification
X = df.drop('annual_income', axis=1)
y = df['annual_income']

# Split the data into training and testing sets (70:30 ratio)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=1)

# Initialize the Random Forest Classifier
rf = RandomForestClassifier(random_state=1)

# Hyperparameter tuning using RandomizedSearchCV
param_dist = {
    'n_estimators': [10, 50, 100, 200, 500],
    'max_depth': [None, 10, 20, 30, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Randomized Search with cross-validation
random_search = RandomizedSearchCV(estimator=rf, param_distributions=param_dist, n_iter=100, cv=3, verbose=2, random_state=1, n_jobs=-1)
random_search.fit(X_train, y_train)

# Print best hyperparameters
print("Best Hyperparameters: ", random_search.best_params_)

# Fit the Random Forest model with the best hyperparameters
best_rf = random_search.best_estimator_
best_rf.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_rf.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Calculate feature importance
feature_importances = pd.DataFrame(best_rf.feature_importances_,
                                   index=X.columns,
                                   columns=['importance']).sort_values('importance', ascending=False)

print("Feature Importance:")
print(feature_importances)


Fitting 3 folds for each of 100 candidates, totalling 300 fits
Best Hyperparameters:  {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 20, 'bootstrap': False}
Accuracy: 0.87
Classification Report:
              precision    recall  f1-score   support

       <=50K       0.90      0.94      0.92      7550
        >50K       0.75      0.64      0.69      2219

    accuracy                           0.87      9769
   macro avg       0.82      0.79      0.80      9769
weighted avg       0.87      0.87      0.87      9769

Feature Importance:
                importance
capital-gain      0.182288
relationship      0.165272
education-num     0.136921
marital-status    0.125537
age               0.093419
hours-per-week    0.063328
fnlwgt            0.050104
capital-loss      0.049395
occupation        0.043633
education         0.040318
workclass         0.019770
sex               0.016574
native-country    0.007376
race              0.006065
